# Same Reality, Different Words
## Step 1 — Preprocessing
**Tool:** spaCy `en_core_web_sm`  
**Input:** `corpus_labor_informality_colombia_180.csv`  
**Output:** `corpus_preprocessed.csv`  

This step applies:
- Tokenization
- Lemmatization
- Stopword removal
- Punctuation and non-alphabetic token removal
- Lowercase normalization

---
### 0. Setup
Run this cell once to install dependencies.

In [1]:
# Run once — install dependencies
!pip install spacy pandas -q
!python -m spacy download en_core_web_sm -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\jaram\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\jaram\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


---
### 1. Load model and corpus

In [3]:
import pandas as pd
import spacy

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded:", nlp.meta["name"], "|", "version:", nlp.meta["version"])

# Load corpus
df = pd.read_csv(r"C:\Users\jaram\Documents\Database_exercises\visual_studio_code\MASTER DCSB\second_semester\text_analysis_nlp\data\corpus_labor_informality_colombia_180.csv")
print(f"\nCorpus shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

spaCy model loaded: core_web_sm | version: 3.8.0

Corpus shape: (180, 4)
Columns: ['text', 'label', 'register', 'prompt_id']


In [4]:
# Check label distribution
print("Label distribution:")
print(df.groupby(["label", "register"])["text"].count().to_string())

Label distribution:
label  register      
0      institutional     60
1      popular_media     60
2      critical_media    60


In [ ]:
# Preview first 5 rows
df.head(5)

,text,label,register,prompt_id
0,"In Colombia, the share of workers lacking form...",0,institutional,A001
1,Social protection coverage among own-account w...,0,institutional,A002
2,Labor formality indicators for the 13 main met...,0,institutional,A003
3,The incidence of informal employment is dispro...,0,institutional,A004
4,Disaggregated data reveal that women in domest...,0,institutional,A005


---
### 2. Define preprocessing function

In [7]:
def preprocess(text):
    """
    Applies spaCy NLP pipeline to a raw text string.
    Returns a clean string of lemmatized, lowercased tokens
    with stopwords, punctuation, spaces, and non-alpha tokens removed.
    """
    doc = nlp(text)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop      # remove stopwords (e.g. "the", "and")
        and not token.is_punct    # remove punctuation
        and not token.is_space    # remove whitespace tokens
        and token.is_alpha        # keep only alphabetic tokens
        and len(token) > 1        # remove single characters
    ]
    return " ".join(tokens)

# Test on one example
example = df["text"].iloc[0]
print("ORIGINAL:", example)
print()
print("CLEAN:   ", preprocess(example))

ORIGINAL: In Colombia, the share of workers lacking formal contract registration remained at 57.3 percent in the most recent survey period, with marked disparities between urban and rural labor markets.

CLEAN:    colombia share worker lack formal contract registration remain percent recent survey period mark disparity urban rural labor market


---
### 3. Apply preprocessing to full corpus

In [8]:
print("Preprocessing 180 texts...")
df["text_clean"] = df["text"].apply(preprocess)
print("Done.")

Preprocessing 180 texts...
Done.


---
### 4. Quality check — sample output per corpus

In [9]:
# Show original vs clean for 2 texts per corpus
for label, register in [(0, "institutional"), (1, "popular_media"), (2, "critical_media")]:
    print(f"\n{'='*60}")
    print(f" {register.upper()}")
    print(f"{'='*60}")
    sample = df[df["label"] == label][["text", "text_clean"]].head(2)
    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print(f"\n  [{i}] ORIGINAL : {row['text'][:100]}...")
        print(f"      CLEAN    : {row['text_clean'][:100]}...")


 INSTITUTIONAL

  [1] ORIGINAL : In Colombia, the share of workers lacking formal contract registration remained at 57.3 percent in t...
      CLEAN    : colombia share worker lack formal contract registration remain percent recent survey period mark dis...

  [2] ORIGINAL : Social protection coverage among own-account workers in the informal sector continues to fall below ...
      CLEAN    : social protection coverage account worker informal sector continue fall statutory threshold pension ...

 POPULAR_MEDIA

  [1] ORIGINAL : Every morning, María sets up her fruit stall before sunrise, knowing that a slow day means skipping ...
      CLEAN    : morning maría set fruit stall sunrise know slow day mean skip dinner sick leave health insurance go ...

  [2] ORIGINAL : Thousands of street vendors in Bogotá work without contracts, relying on whatever they sell that day...
      CLEAN    : thousand street vendor bogotá work contract rely sell day cover rent food transportation leave emerg

---
### 5. Token count statistics

In [10]:
df["token_count_original"] = df["text"].apply(lambda x: len(x.split()))
df["token_count_clean"]    = df["text_clean"].apply(lambda x: len(x.split()))

stats = df.groupby("register")[["token_count_original", "token_count_clean"]].mean().round(1)
stats.columns = ["avg_tokens_original", "avg_tokens_clean"]
stats["reduction_%"] = ((stats["avg_tokens_original"] - stats["avg_tokens_clean"]) 
                         / stats["avg_tokens_original"] * 100).round(1)
print(stats.to_string())

overall_reduction = ((df["token_count_original"].mean() - df["token_count_clean"].mean()) 
                      / df["token_count_original"].mean() * 100)
print(f"\nOverall average token reduction: {overall_reduction:.1f}%")

                avg_tokens_original  avg_tokens_clean  reduction_%
register                                                          
critical_media                 39.3              24.1         38.7
institutional                  26.5              18.2         31.3
popular_media                  28.8              15.0         47.9

Overall average token reduction: 39.6%


---
### 6. Save preprocessed corpus

In [11]:
output_cols = [
    "text", "text_clean", "label", "register", "prompt_id",
    "token_count_original", "token_count_clean"
]

df[output_cols].to_csv("corpus_preprocessed.csv", index=False)
print("Saved: corpus_preprocessed.csv")
print(f"Shape: {df[output_cols].shape}")
print("\nStep 1 complete. Ready for Step 2 — POS Tagging.")

Saved: corpus_preprocessed.csv
Shape: (180, 7)

Step 1 complete. Ready for Step 2 — POS Tagging.


In [12]:
# Final preview of output
df[output_cols].head()

,text,text_clean,label,register,prompt_id,token_count_original,token_count_clean
0,"In Colombia, the share of workers lacking form...",colombia share worker lack formal contract reg...,0,institutional,A001,29,18
1,Social protection coverage among own-account w...,social protection coverage account worker info...,0,institutional,A002,25,18
2,Labor formality indicators for the 13 main met...,labor formality indicator main metropolitan ar...,0,institutional,A003,23,15
3,The incidence of informal employment is dispro...,incidence informal employment disproportionate...,0,institutional,A004,24,16
4,Disaggregated data reveal that women in domest...,disaggregate datum reveal woman domestic servi...,0,institutional,A005,25,18
